# Agent Pipeline Benchmark & Analysis

Evaluating the multi-agent Research → Code → Review system's performance,
cost profile, and failure modes across a suite of programming tasks.

---
**Prerequisites:** Run `pip install -r requirements.txt` and set `GROQ_API_KEY` in `.env`.

In [ ]:
from __future__ import annotations
import sys, os, json, time, uuid
from pathlib import Path
from datetime import datetime, timezone

# Ensure project root is on the path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

from graph.build_graph import build_graph
from graph.state import new_state, GraphState
from graph.supervisor import route_after_review
from sandbox.executor import run_python_code

print("All imports OK")
print(f"Python: {sys.version}")
print(f"GROQ_API_KEY set: {bool(os.environ.get('GROQ_API_KEY'))}")

## 1. Graph Architecture

Let's inspect the compiled state machine to understand the topology.

In [ ]:
app = build_graph()

print(f"Nodes: {len(app.nodes)}")
for name, node in app.nodes.items():
    print(f"  ├─ {name}")

print(f"\nEdges: {len(app.edges)}")
for edge in app.edges:
    print(f"  ├─ {edge}")

# Visualize the graph as mermaid
png_bytes = app.get_graph().draw_mermaid_png()
with open("architecture.png", "wb") as f:
    f.write(png_bytes)
print("\nGraph diagram saved to architecture.png")

## 2. State Model

The typed `GraphState` is the single source of truth passed between nodes.

In [ ]:
state = new_state("Write a function that returns the nth Fibonacci number.", max_iterations=3)

print("Initial state:")
for key, value in sorted(state.items()):
    print(f"  {key:20s} = {value!r}")

## 3. Supervisor Routing Logic

The `route_after_review` conditional edge is the only branch point. Let's verify the routing matrix.

In [ ]:
from graph.state import TestResult

scenarios = [
    ("Tests pass", [TestResult(passed=True, output="ok", error=None)], 1, "human_approval"),
    ("Tests fail, retries left", [TestResult(passed=False, output="", error="fail")], 1, "coder"),
    ("Tests fail, no retries left", [TestResult(passed=False, output="", error="fail")], 3, "human_approval"),
]

print("| Scenario | Next Node |")
print("|----------|-----------|")
for name, test_results, iteration, expected in scenarios:
    s = new_state("dummy", max_iterations=3)
    s["test_results"] = test_results
    s["iteration_count"] = iteration
    result = route_after_review(s)
    status = "✅" if result == expected else "❌"
    print(f"| {name} | {result} {status} |")

## 4. Sandbox Execution

The sandbox executor runs generated code in isolation. Let's verify it works correctly.

In [ ]:
# Passing code
result = run_python_code("assert 1 + 1 == 2\nprint('hello from sandbox')")
print(f"PASSING CODE:")
print(f"  passed: {result.passed}")
print(f"  stdout: {result.stdout.strip()}")
print(f"  used_docker: {result.used_docker}")

print()

# Failing code
result = run_python_code("assert 1 == 2, 'intentional failure'")
print(f"FAILING CODE:")
print(f"  passed: {result.passed}")
print(f"  stderr: {result.stderr.strip()[:100]}")

## 5. End-to-End Pipeline Run

Execute the full pipeline on a simple task and inspect the output.
**Note:** This requires a valid `GROQ_API_KEY` and will consume tokens.

In [ ]:
if os.environ.get("GROQ_API_KEY"):
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    
    print(f"Thread ID: {thread_id}")
    print("Running pipeline...")
    
    start = time.monotonic()
    output = app.invoke(new_state("Write a function to reverse a string. Include assert tests."), config=config)
    elapsed = time.monotonic() - start
    
    print(f"Completed in {elapsed:.1f}s")
    
    # Check if we hit the interrupt
    app_state = app.get_state(config)
    if app_state.next and "human_approval" in app_state.next:
        from langgraph.types import Command
        interrupt_val = app_state.tasks[0].interrupts[0].value
        print(f"\nPaused at human_approval gate (auto-approving for benchmark)")
        output = app.invoke(Command(resume={"approved": True, "feedback": None}), config=config)
        print(f"\nFinal report:")
        print(output.get("final_report", "(no report)")[:2000])
else:
    print("SKIPPED: Set GROQ_API_KEY in .env to run the pipeline end-to-end.")

## 6. Results Analysis

If you ran the benchmark via `python scripts/run_benchmark.py`, the results
are in `reports/benchmark_results.json`. Let's load and analyze them.

In [ ]:
results_path = Path("reports/benchmark_results.json")
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    
    total = len(results)
    passed = sum(1 for r in results if r["tests_passed"])
    approved = sum(1 for r in results if r["human_approved"])
    total_tokens = sum(r.get("total_tokens", 0) for r in results)
    avg_iter = sum(r.get("iterations_used", 0) for r in results) / total if total else 0
    
    print(f"=== BENCHMARK SUMMARY ===")
    print(f"Tasks: {total}")
    print(f"Success rate: {passed}/{total} ({passed/total*100:.0f}%)")
    print(f"Human approval rate: {approved}/{total}")
    print(f"Avg iterations: {avg_iter:.1f}")
    print(f"Total tokens: {total_tokens:,}")
    print()
    
    print(f"{'Task':<20} {'Passed':<8} {'Iters':<6} {'Tokens':<10} {'Duration':<10}")
    print("-" * 54)
    for r in results:
        status = "✅" if r["tests_passed"] else "❌"
        print(f"{r['task_id']:<20} {status:<8} {r.get('iterations_used', 0):<6} {r.get('total_tokens', 0):<10,} {r.get('duration_seconds', 0):<10.1f}")
else:
    print("No benchmark results found.")
    print("Run `python scripts/run_benchmark.py --quick` to generate them.")

## 7. Key Observations

Based on the system architecture and benchmark results:

| Observation | Implication |
|-------------|-------------|
| Cost-aware model routing (8B for research, 70B+ for code) | Research is the high-volume, low-cognitive-load phase — using a smaller model saves ~4× tokens |
| Provider fallback (Groq → OpenAI) | System tolerates provider outages without failing the node |
| Bounded retries (default 3) | Prevents infinite loops; human sees last failing state |
| MCP-based tool discovery | Tools are swappable without agent code changes |
| Sandboxed Docker execution | Generated code can't harm the host or access network |

### Failure Mode Analysis

Common failure patterns to track:
1. **Test assertion mismatch** — generated code has a logic bug, coder fixes on retry
2. **Timeout** — code enters an infinite loop, sandbox kills after 20s
3. **Human rejection** — code looks correct but reviewer spots a subtle issue

Tracking these over time lets you improve prompts, adjust model choice, and tune iteration limits.

## 8. Conclusion

The Research → Code → Review multi-agent pipeline demonstrates:

- **State machine design** using LangGraph
- **Real MCP protocol integration** for tool discovery
- **Safety-first design** — sandboxed execution, path jailing, human-in-the-loop
- **Cost-aware architecture** — model routing by node role
- **Resilience patterns** — provider fallbacks, bounded retries, checkpointing

Measured across N benchmark tasks, the system's self-test and reference-test pass rates are recorded
self-test and reference-test pass rates. Re-run with `python scripts/run_benchmark.py`.